In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_global_tech_gold.01_dims_tables;


In [0]:
from pyspark.sql.functions import lit, when, upper, trim, col, date_format, year, month, quarter, dayofmonth, dayofweek, weekofyear, sequence, explode, coalesce, last_day, add_months, to_date, concat_ws

In [0]:
accounts_df = spark.table("02_global_tech_silver.transformed_tables.transformed_accounts")
pay_df = spark.table("02_global_tech_silver.transformed_tables.transformed_payroll")
gl_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_general_ledgers")
emp_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_employee")
dept_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_departments")
company_df = spark.table("02_global_tech_silver.transformed_tables.transformed_company")


In [0]:
# Date Dimension
min_gl_date = gl_df.agg({"entry_date": "min"}).collect()[0][0]
max_gl_date = gl_df.agg({"entry_date": "max"}).collect()[0][0]
min_pay_date = pay_df.agg({"pay_date": "min"}).collect()[0][0]
max_pay_date = pay_df.agg({"pay_date": "max"}).collect()[0][0]

all_dates = [min_gl_date, max_gl_date, min_pay_date, max_pay_date]
start_date = min([d for d in all_dates if d is not None])
end_date   = max([d for d in all_dates if d is not None])

dim_date = (
    spark.createDataFrame([(start_date, end_date)], ["start", "end"])
        .withColumn("date", explode(sequence(col("start"), col("end"))))
        .drop("start", "end")
        .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
        .withColumn("year", year(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
        .withColumn("day_of_month", dayofmonth(col("date")))
        .withColumn("day_of_week", dayofweek(col("date")))
        .withColumn("week_of_year", weekofyear(col("date")))
)

In [0]:
# Company dimension
dim_company = company_df.select(
    col("company_sk").alias("company_key"),
    col("company_id"),
    col("company_name"),
    col("industry"),
    col("country"),
    col("established_date"),
    col("company_is_active")
)

# display(dim_company.columns)

In [0]:
# Department dimension

dim_department = dept_df.select(
    col("department_sk").alias("department_key"),
    col("department_id"),
    col("department_code"),
    col("department_name"),
    col("cost_center")
)

In [0]:
# Employee dimension

dim_employee = (
    emp_df.alias("e")
    .join(dept_df.alias("d"), col("e.department_id") == col("d.department_id"), "left")
    .join(company_df.alias("c"), col("e.company_id") == col("c.company_id"), "left")
    .select(
        col("e.employee_sk").alias("employee_key"),
        col("e.employee_id"),
        col("e.employee_code"),
        col("e.first_name"),
        col("e.last_name"),
        col("e.email"),
        col("d.department_sk").alias("department_key"),
        col("c.company_sk").alias("company_key"),
        col("e.position"),
        col("e.hire_date"),
        col("e.termination_date"),
        col("e.employee_is_active"),
        col("e.tenure_days")
    )
)

In [0]:
# Accounts Dimension
dim_account = accounts_df.select(
    col("account_sk").alias("account_key"),
    col("account_id"),
    col("account_code"),
    col("account_name"),
    col("account_type"),
    col("category"),
    col("account_is_active")
)

In [0]:
# Save Dimension Tables
dim_date.write.mode("overwrite").saveAsTable("03_global_tech_gold.01_dims_tables.dim_date")
dim_company.write.mode("overwrite").saveAsTable("03_global_tech_gold.01_dims_tables.dim_company")
dim_department.write.mode("overwrite").saveAsTable("03_global_tech_gold.01_dims_tables.dim_department")
dim_employee.write.mode("overwrite").saveAsTable("03_global_tech_gold.01_dims_tables.dim_employee")
dim_account.write.mode("overwrite").saveAsTable("03_global_tech_gold.01_dims_tables.dim_account")